# WeaveGuard AI — Étape B2 : contrôle des quasi-doublons et dataset v2
**Pourquoi** : dans le v1, le seuil de quasi-doublons est tombé à -1 (aucune fusion) alors qu'il reste des paires d'images très proches entre `train`, `val` et `test`.
Ce notebook :
1. recalcule un **hash perceptuel fin (256 bits)** sur les images du v1 ;
2. te montre **visuellement** des paires proches entre partitions, par niveau de distance (c'est toi qui juges si ce sont de vraies quasi-copies) ;
3. **regroupe** les quasi-doublons, refait le split (même méthode, mêmes garde-fous) et exporte un **dataset v2** + zip sur Drive.

Le v1 n'est pas modifié (il reste sur Drive pour la traçabilité).

In [ ]:
# 1) Drive + décompression du v1 en local
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# 2) Configuration
from pathlib import Path
import json, hashlib, io, shutil, random, datetime, warnings, zipfile
import numpy as np, pandas as pd, yaml
from PIL import Image, ImageOps
from concurrent.futures import ThreadPoolExecutor
warnings.filterwarnings('ignore')

ROOT = Path('/content/drive/MyDrive/WeaveGuard')
DRIVE_OUT = ROOT / 'datasets'
V1 = Path('/content/weaveguard_dataset_v1')
V2 = Path('/content/weaveguard_dataset_v2')
SEED = 42
SPLITS = ('train', 'val', 'test')
N_FOLDS = 20
FOLDS = {'train': 14, 'val': 3, 'test': 3}
T256_OVERRIDE = None          # None = choix automatique ; sinon un entier (distance de Hamming sur 256 bits)
T_GRID = [-1, 0, 2, 4, 6, 8, 10, 12, 16, 20, 24, 32]
MAX_COMP_FRAC = 0.10
N_SEEDS_SEARCH = 300
MIN_IMG_PER_CLASS_EVAL = 8
CROP = 512
CROPS_PER_IMAGE = {'train': 4, 'val': 3, 'test': 3}
N_THREADS = 16
BINS = [(0, 5), (6, 12), (13, 20), (21, 30), (31, 45)]
RES_BOX = getattr(Image, 'Resampling', Image).BOX

if not V1.exists():
    zipfile.ZipFile(DRIVE_OUT / 'weaveguard_dataset_v1.zip').extractall(V1)
print('v1 local :', V1.exists())

In [ ]:
# 3) Chargement du v1 + hash 256 bits
man = pd.read_csv(V1 / 'manifest.csv')
y = yaml.safe_load(open(V1 / 'data.yaml', encoding='utf-8'))
NAMES = [y['names'][k] for k in sorted(y['names'], key=int)]; NC = len(NAMES)

def load_item(r):
    name, split = r['name'], r['split']
    with Image.open(V1 / 'images' / split / name) as im:
        g = np.asarray(im.convert('L').resize((17, 16), RES_BOX), dtype=np.int16)
    h = np.packbits((g[:, 1:] > g[:, :-1]).flatten())
    boxes = []
    for line in (V1 / 'labels' / split / (Path(name).stem + '.txt')).read_text().splitlines():
        s = line.split()
        if len(s) == 5: boxes.append((int(s[0]), *map(float, s[1:])))
    return h, boxes

with ThreadPoolExecutor(N_THREADS) as ex:
    items = list(ex.map(load_item, man.to_dict('records')))
H = np.stack([it[0] for it in items])
df = man.copy()
df['old_split'] = df['split']
df['boxes'] = [it[1] for it in items]
df['classes_idx'] = df.boxes.map(lambda b: sorted(set(x[0] for x in b)))
n = len(df); print(n, 'images chargées,', H.shape[1] * 8, 'bits de hash')

tab8 = np.array([bin(i).count('1') for i in range(256)], dtype=np.uint8)
D = np.zeros((n, n), dtype=np.uint16)
for i in range(n - 1):
    D[i, i + 1:] = tab8[H[i + 1:] ^ H[i]].sum(axis=1)
D = D + D.T
iu = np.triu_indices(n, 1); dvals = D[iu]
old = df.old_split.values
cross = old[iu[0]] != old[iu[1]]
rows = [dict(distance=f'{a}-{b}', paires_total=int(((dvals >= a) & (dvals <= b)).sum()),
             entre_partitions_v1=int(((dvals >= a) & (dvals <= b) & cross).sum())) for a, b in BINS]
print('Paires proches selon la distance (256 bits) :'); display(pd.DataFrame(rows))

In [ ]:
# 4) Vérification VISUELLE : paires entre partitions (v1), par niveau de distance
import matplotlib.pyplot as plt
def thumb(i, size=230):
    r = df.iloc[i]
    im = ImageOps.exif_transpose(Image.open(V1 / 'images' / r.old_split / r['name'])).convert('RGB')
    im.thumbnail((size, size)); return im

def show_pairs(pairs, title, fname=None, per_row=3):
    if not pairs: print('(aucune paire)'); return
    nr = (len(pairs) + per_row - 1) // per_row
    fig, axes = plt.subplots(nr, per_row * 2, figsize=(per_row * 5.2, nr * 2.9))
    axes = np.atleast_2d(axes)
    for ax in axes.flat: ax.axis('off')
    for k, (i, j, d) in enumerate(pairs):
        r, c = divmod(k, per_row)
        for off, idx in ((0, i), (1, j)):
            ax = axes[r, c * 2 + off]; ax.imshow(thumb(idx))
            ax.set_title(f"{df.iloc[idx].split[:5]} | {df.iloc[idx]['name'][:18]}" + (f'\nd={d}' if off == 0 else ''), fontsize=7)
    plt.suptitle(title); plt.tight_layout()
    if fname: plt.savefig(DRIVE_OUT / fname, dpi=110)
    plt.show()

rng = random.Random(SEED)
for a, b in BINS:
    idx = np.where((dvals >= a) & (dvals <= b) & cross)[0]
    pick = rng.sample(list(idx), min(6, len(idx)))
    pairs = [(int(iu[0][k]), int(iu[1][k]), int(dvals[k])) for k in pick]
    show_pairs(pairs, f'Paires ENTRE partitions v1, distance {a}-{b} ({len(idx)} paires au total)', f'pairs_v1_{a}_{b}.png')

In [ ]:
# 5) Regroupement : photo source + doublon exact + quasi-doublon (hash 256 bits)
def components(T):
    parent = list(range(n))
    def find(a):
        while parent[a] != a:
            parent[a] = parent[parent[a]]; a = parent[a]
        return a
    def union(a, b):
        ra, rb = find(a), find(b)
        if ra != rb: parent[rb] = ra
    for key in (df.photo_id, df.md5):
        first = {}
        for i, k in enumerate(key):
            if k in first: union(i, first[k])
            else: first[k] = i
    if T >= 0:
        for k in np.where(dvals <= T)[0]: union(int(iu[0][k]), int(iu[1][k]))
    return np.array([find(i) for i in range(n)])

rows = []
for T in T_GRID:
    sizes = pd.Series(components(T)).value_counts()
    rows.append(dict(seuil_256=T, n_groupes=len(sizes), plus_gros_groupe=int(sizes.iloc[0]), part=round(sizes.iloc[0] / n, 3)))
tbl = pd.DataFrame(rows); display(tbl)
okT = tbl[tbl.part <= MAX_COMP_FRAC]
T_USED = T256_OVERRIDE if T256_OVERRIDE is not None else int(okT.seuil_256.max())
df['group'] = components(T_USED)
print(f'>>> Seuil retenu (256 bits) : {T_USED} -> {df.group.nunique()} groupes')
if df.group.nunique() < 3 * N_FOLDS: print('[!] Peu de groupes : split moins équilibré.')

In [ ]:
# 6) Nouveau split stratifié par groupes
from sklearn.model_selection import StratifiedGroupKFold
P = np.zeros((n, NC), dtype=int)
for i, cl in enumerate(df.classes_idx):
    for c in cl: P[i, c] = 1
cls_freq = P.sum(0); tot = cls_freq.astype(float)
gcls = df.groupby('group').classes_idx.apply(lambda s: set().union(*[set(x) for x in s]))
df['strat'] = df.group.map(lambda g: min(gcls[g], key=lambda c: cls_freq[c]))
target = np.array([FOLDS[s] / N_FOLDS for s in SPLITS])
order = [s for s in SPLITS for _ in range(FOLDS[s])]
best = None
for seed in range(SEED, SEED + N_SEEDS_SEARCH):
    fold = np.zeros(n, dtype=int)
    sg = StratifiedGroupKFold(n_splits=N_FOLDS, shuffle=True, random_state=seed)
    for k, (_, te) in enumerate(sg.split(np.zeros(n), df.strat.values, df.group.values)): fold[te] = k
    perm = np.random.RandomState(seed).permutation(N_FOLDS)
    fsplit = {int(perm[i]): order[i] for i in range(N_FOLDS)}
    sp = np.array([fsplit[f] for f in fold])
    cnt = np.zeros((NC, 3)); score = 0.0
    for si, s in enumerate(SPLITS):
        cnt[:, si] = P[sp == s].sum(0); score += 2 * abs((sp == s).mean() - target[si])
    score += np.abs(cnt / tot[:, None] - target[None, :]).sum()
    score += 10 * int(((cnt[:, 1] < MIN_IMG_PER_CLASS_EVAL) | (cnt[:, 2] < MIN_IMG_PER_CLASS_EVAL)).sum())
    if best is None or score < best[0]: best = (score, seed, sp)
score, BEST_SEED, sp = best
df['split'] = sp
print(f'Meilleure graine : {BEST_SEED} (score {score:.3f})')

In [ ]:
# 7) Vérifications v2
import matplotlib.pyplot as plt
assert df.groupby('group').split.nunique().max() == 1
assert df.groupby('md5').split.nunique().max() == 1
new = df.split.values; cross2 = new[iu[0]] != new[iu[1]]
rows = [dict(distance=f'{a}-{b}', entre_partitions_v1=int(((dvals >= a) & (dvals <= b) & cross).sum()),
             entre_partitions_v2=int(((dvals >= a) & (dvals <= b) & cross2).sum())) for a, b in BINS]
print('Paires proches entre partitions : v1 vs v2'); display(pd.DataFrame(rows))
below = int(((dvals <= T_USED) & cross2).sum())
print(f'Paires <= {T_USED} entre partitions en v2 : {below} (doit être 0)'); assert below == 0

print('\nImages par partition :'); print(df.split.value_counts().reindex(SPLITS).to_string())
img_cls = pd.DataFrame({s: P[(df.split == s).values].sum(0) for s in SPLITS}, index=NAMES); img_cls['total'] = img_cls.sum(axis=1)
display(img_cls)
low = img_cls[(img_cls.val < MIN_IMG_PER_CLASS_EVAL) | (img_cls.test < MIN_IMG_PER_CLASS_EVAL)]
if len(low): print('[!] Classes avec peu d images en val/test :', list(low.index))
box_rows = [dict(split=r.split, cls=NAMES[b[0]]) for r in df.itertuples() for b in r.boxes]
box_cls = pd.DataFrame(box_rows).groupby(['cls', 'split']).size().unstack(fill_value=0).reindex(columns=list(SPLITS), fill_value=0)
box_cls['total'] = box_cls.sum(axis=1)

# paires les plus proches qui restent entre partitions en v2 (transparence)
idx = np.where(cross2)[0]; idx = idx[np.argsort(dvals[idx])][:6]
show_pairs([(int(iu[0][k]), int(iu[1][k]), int(dvals[k])) for k in idx], 'v2 : paires les PLUS proches restant entre partitions', 'pairs_v2_closest.png')
img_cls[list(SPLITS)].plot(kind='bar', stacked=True, figsize=(9, 4), title='v2 : images par classe et partition')
plt.tight_layout(); plt.savefig(DRIVE_OUT / 'split_classes_v2.png', dpi=120); plt.show()

In [ ]:
# 8) Export du dataset v2 (copie depuis le v1)
if V2.exists(): shutil.rmtree(V2)
for s in SPLITS:
    (V2 / 'images' / s).mkdir(parents=True); (V2 / 'labels' / s).mkdir(parents=True)
def export(r):
    shutil.copy2(V1 / 'images' / r['old_split'] / r['name'], V2 / 'images' / r['split'] / r['name'])
    stem = Path(r['name']).stem
    shutil.copy2(V1 / 'labels' / r['old_split'] / (stem + '.txt'), V2 / 'labels' / r['split'] / (stem + '.txt'))
with ThreadPoolExecutor(N_THREADS) as ex: list(ex.map(export, df[['name', 'old_split', 'split']].to_dict('records')))
yaml.safe_dump(dict(path=str(V2), train='images/train', val='images/val', test='images/test', nc=NC,
                    names={i: nm for i, nm in enumerate(NAMES)}), open(V2 / 'data.yaml', 'w', encoding='utf-8'), sort_keys=False)
for s in SPLITS: print(s, len(list((V2 / 'images' / s).iterdir())), 'images,', len(list((V2 / 'labels' / s).iterdir())), 'labels')

In [ ]:
# 9) Recadrages sans défaut pour PatchCore (chaque partition utilise ses propres images)
def make_crops(split):
    out_dir = V2 / 'normal_crops' / split; out_dir.mkdir(parents=True, exist_ok=True)
    count = skipped = 0
    for p in sorted((V2 / 'images' / split).iterdir()):
        im = ImageOps.exif_transpose(Image.open(p)).convert('RGB'); W, H_ = im.size
        if min(W, H_) < CROP: skipped += 1; continue
        bxs = []
        for line in (V2 / 'labels' / split / (p.stem + '.txt')).read_text().splitlines():
            s = line.split()
            if len(s) != 5: continue
            cx, cy, bw, bh = map(float, s[1:])
            x0, y0, x1, y1 = (cx-bw/2)*W, (cy-bh/2)*H_, (cx+bw/2)*W, (cy+bh/2)*H_
            m = max(32, 0.15 * max(x1 - x0, y1 - y0)); bxs.append((x0 - m, y0 - m, x1 + m, y1 + m))
        rg = random.Random(f'{SEED}-{p.name}'); got = []; tries = 0
        while len(got) < CROPS_PER_IMAGE[split] and tries < 60:
            tries += 1
            x, y_ = rg.randint(0, W - CROP), rg.randint(0, H_ - CROP)
            if any(not (x + CROP <= a or x >= c or y_ + CROP <= b or y_ >= d) for a, b, c, d in bxs): continue
            if any(max(0, min(x, gx) + CROP - max(x, gx)) * max(0, min(y_, gy) + CROP - max(y_, gy)) > 0.3 * CROP * CROP for gx, gy in got): continue
            got.append((x, y_))
        for k, (x, y_) in enumerate(got): im.crop((x, y_, x + CROP, y_ + CROP)).save(out_dir / f'{p.stem}_c{k}.jpg', quality=95)
        count += len(got)
    return count, skipped
crop_stats = {}
for s in SPLITS:
    c, sk = make_crops(s); crop_stats[s] = dict(crops=c, images_too_small=sk)
print('Recadrages sans défaut :', crop_stats)
files = sorted((V2 / 'normal_crops' / 'train').glob('*.jpg')); random.Random(SEED).shuffle(files)
fig, axes = plt.subplots(4, 6, figsize=(14, 10))
for ax, f in zip(axes.flat, files[:24]): ax.imshow(Image.open(f).resize((200, 200))); ax.axis('off')
plt.suptitle('24 recadrages sans défaut (train v2) : vérifie qu aucun défaut visible ne subsiste'); plt.tight_layout(); plt.show()

In [ ]:
# 10) Manifeste v2, version, zip Drive, rapport à coller dans le chat
man2 = df[['name', 'split', 'group', 'photo_id', 'md5', 'w', 'h', 'n_boxes']].copy()
man2['old_split_v1'] = df.old_split
man2['classes'] = df.classes_idx.map(lambda c: ';'.join(NAMES[i] for i in c))
man2.to_csv(V2 / 'manifest.csv', index=False)
ds_hash = hashlib.sha256('\n'.join(sorted(f'{a}|{b}|{c}' for a, b, c in zip(man2.name, man2.md5, man2.split))).encode()).hexdigest()
parent = None
try: parent = json.load(open(DRIVE_OUT / 'dataset_stats_v1.json'))['dataset_hash_sha256']
except Exception: pass
stats = dict(dataset_version='v2', dataset_hash_sha256=ds_hash, parent_dataset_hash_v1=parent,
             created=datetime.datetime.now().isoformat(timespec='seconds'),
             source='textile_defects (v1 regroupé par quasi-doublons)', seed_split=int(BEST_SEED),
             near_dup_hash='dhash 256 bits (17x16, BOX)', near_dup_hamming_threshold_256=int(T_USED),
             n_groups=int(df.group.nunique()), n_images=int(n), images_per_split={s: int((df.split == s).sum()) for s in SPLITS},
             images_per_class=img_cls.to_dict('index'), boxes_per_class=box_cls.to_dict('index'),
             moved_vs_v1=int((df.split != df.old_split).sum()),
             near_pairs_across_splits={f'{a}-{b}': dict(v1=int(((dvals >= a) & (dvals <= b) & cross).sum()),
                                                        v2=int(((dvals >= a) & (dvals <= b) & cross2).sum())) for a, b in BINS},
             normal_crops=dict(size_px=CROP, per_split=crop_stats), classes=NAMES)
(V2 / 'dataset_stats.json').write_text(json.dumps(stats, indent=1, ensure_ascii=False), encoding='utf-8')
for src, dst in ((V2 / 'dataset_stats.json', 'dataset_stats_v2.json'), (V2 / 'manifest.csv', 'manifest_v2.csv'), (V2 / 'data.yaml', 'data_v2.yaml')):
    shutil.copy2(src, DRIVE_OUT / dst)
zp = shutil.make_archive(str(DRIVE_OUT / 'weaveguard_dataset_v2'), 'zip', V2)
print('Zip sauvegardé sur Drive :', zp); print('=' * 70)
print(json.dumps(stats, indent=1, ensure_ascii=False)[:9000])
try:
    from google.colab import files; files.download(str(DRIVE_OUT / 'dataset_stats_v2.json'))
except Exception: pass